# From Tokens to a Tiny Language Model

This notebook follows one continuous path through a modern decoder-only Transformer:

`token IDs → embedding → attention → RMSNorm → RoPE → SwiGLU → logits → loss`

Run each cell in order. Observe the output first, then change a parameter and predict what will happen. The companion implementation lives in `transformer_tutorial.py`.

In [ ]:
# This supports launching Jupyter from either the repository root or notebooks/.
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'transformer_tutorial.py').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import math
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)
print('PyTorch:', torch.__version__)
print('Repository root:', repo_root)

## 1. Embedding: an ID is an address, not a quantity

### Why token IDs are not enough

A tokenizer converts text into integer IDs. Suppose a tiny vocabulary contains:

```text
0 → <pad>
1 → I
2 → love
3 → cats
4 → dogs
```

The sentence `I love cats` becomes `[1, 2, 3]`. These numbers are arbitrary labels, like student ID numbers. ID 3 is not three times ID 1, and the numerical distance between IDs does not express semantic similarity. A neural network therefore needs a richer representation than the raw IDs.

### An embedding is a learnable lookup table

`nn.Embedding(num_embeddings=8, embedding_dim=4)` creates a learnable table with eight rows and four columns:

```text
                    four learnable values
token 0  ──────→  [ ?,  ?,  ?,  ? ]
token 1  ──────→  [ ?,  ?,  ?,  ? ]
token 2  ──────→  [ ?,  ?,  ?,  ? ]
token 3  ──────→  [ ?,  ?,  ?,  ? ]
token 4  ──────→  [ ?,  ?,  ?,  ? ]
token 5  ──────→  [ ?,  ?,  ?,  ? ]
token 6  ──────→  [ ?,  ?,  ?,  ? ]
token 7  ──────→  [ ?,  ?,  ?,  ? ]
```

The table has shape `[V, D]`, where `V` is the vocabulary size and `D` is the embedding dimension. It contains $V\times D$ trainable parameters. Real language models may use tens of thousands of rows and thousands of columns; this example uses `[8, 4]` only so that we can inspect it easily.

### What does `embedding(token_ids)` do?

Given the input IDs `[1, 3, 1, 6]`, PyTorch selects rows 1, 3, 1, and 6 from the table:

```text
token IDs:       [ 1,  3,  1,  6 ]
                    │   │   │   │
embedding lookup:  row row row row
                    1   3   1   6
                    │   │   │   │
vectors:         [ v₁, v₃, v₁, v₆ ]
```

This changes the tensor shape from `[B, T]` to `[B, T, D]`:

```text
token_ids                         embedded vectors
shape [1, 4]                      shape [1, 4, 4]
                                  
[[1, 3, 1, 6]]   ── lookup ──→   [[[...],  token 1
                                    [...],  token 3
                                    [...],  token 1
                                    [...]]] token 6
```

Here `B=1` means one sequence in the batch, `T=4` means four tokens in the sequence, and `D=4` means four learned features per token. Because ID 1 appears twice, the lookup returns exactly the same initial vector twice.

### Where does meaning come from?

The vectors start as random numbers. During language-model training, prediction errors flow backward through the Transformer and update the embedding table:

```text
text → token IDs → embedding vectors → Transformer → next-token prediction
                       ↑                                │
                       └──── parameter updates ── loss ─┘
```

Tokens used in similar contexts gradually acquire useful related representations. The individual dimensions do not usually have simple labels such as “animal” or “size”; meaning is distributed across the whole vector.

The same token always receives the same **initial** embedding, but later Transformer layers combine it with position and context. Consequently, the word `apple` can develop different contextual representations in `eat an apple` and `Apple released a computer`.

In short:

```text
Tokenizer:    text → token IDs
Embedding:    token IDs → learnable vectors
Transformer:  vectors → context-aware vectors
LM head:      context-aware vectors → next-token scores
```

In [ ]:
# Eight tokens, with one four-dimensional vector per token.
embedding = nn.Embedding(num_embeddings=8, embedding_dim=4)

# Shape: [batch_size=1, sequence_length=4]
token_ids = torch.tensor([[1, 3, 1, 6]])

# Embedding lookup changes [B, T] into [B, T, D].
x = embedding(token_ids)

print('Embedding table shape:', tuple(embedding.weight.shape))
print('Token ID shape:       ', tuple(token_ids.shape))
print('Output shape:         ', tuple(x.shape))
print('\nToken IDs:', token_ids)
print('\nSelected vectors:', x)
print('\nThe two ID=1 vectors are equal:', torch.equal(x[0, 0], x[0, 2]))

## 2. Causal Attention: only read the past

### The search-engine intuition

Attention is easiest to understand as a small search engine. Every token creates three different vectors from the same input representation:

- **Query (Q):** what information is this token looking for?
- **Key (K):** what kind of information does this token contain?
- **Value (V):** what information should this token contribute if selected?

A web search has the same separation:

```text
your search text       → Query
page title / keywords  → Key
actual page contents   → Value
```

The search engine compares your query with page keys, then returns the contents of the best-matching pages. Attention compares Q with K, then combines the corresponding V vectors.

### A sentence-level example

Consider the partial sentence:

```text
The  cat  sat
 0    1    2
```

When processing `sat`, its query might encode something like “Which earlier token is the thing performing this action?” The key produced by `cat` may strongly advertise “I am a noun and a possible subject,” while the key from `The` is a poorer match. The resulting attention weights might look like:

```text
query from 'sat'
       │
       ├── compare with key('The') ──→ score 0.1 ──→ weight 0.1
       ├── compare with key('cat') ──→ score 2.4 ──→ weight 0.8
       └── compare with key('sat') ──→ score 0.3 ──→ weight 0.1

output for 'sat'
    = 0.1 × value('The')
    + 0.8 × value('cat')
    + 0.1 × value('sat')
```

This example is only an intuition—the model learns its own distributed features rather than explicit labels such as “subject.” Different attention heads can learn different matching rules.

### Why are K and V separate?

A key is optimized for **being found**, while a value is optimized for **providing useful content**. A library analogy makes the distinction clear:

```text
book-catalog keywords → Key: helps decide which book is relevant
book contents         → Value: supplies the information you actually read
```

The catalog entry and the book serve different purposes, even though they refer to the same item. Likewise, K determines attention weights, while V determines what information flows into the output.

### Where do Q, K, and V come from?

They are three learned linear projections of the same input representation $X$:

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V$$

```text
                         ┌── × W_Q ──→ Q: what am I looking for?
input representation X ─┼── × W_K ──→ K: how should I be matched?
                         └── × W_V ──→ V: what content do I provide?
```

At the beginning of training, the projection matrices $W_Q$, $W_K$, and $W_V$ are randomly initialized using a controlled initialization scheme. Q, K, and V are **not independently sampled random vectors**. They are deterministic results computed from the current input and the current projection matrices. The same $X$ and unchanged weights always produce the same Q, K, and V.

### How training changes them

The persistent learned objects are the projection matrices. Q, K, and V are temporary activations created during each forward pass:

```text
saved model parameters                 temporary forward results
──────────────────────                 ─────────────────────────
W_Q, W_K, W_V          ── applied to X ──→ Q, K, V
updated by optimizer                      recomputed every forward pass
```

A training step follows this loop:

```text
token IDs
    ↓
embedding → X
    ↓
Q = XW_Q,  K = XW_K,  V = XW_V
    ↓
attention → Transformer → next-token logits
    ↓
compare with the correct next token → loss
    ↓
backward: compute ∂loss/∂W_Q, ∂loss/∂W_K, ∂loss/∂W_V
    ↓
optimizer: slightly update W_Q, W_K, and W_V
    ↓
next forward pass produces improved Q, K, and V
```

For example, gradient descent updates $W_K$ as:

$$W_K \leftarrow W_K-\eta\frac{\partial L}{\partial W_K}$$

Here $L$ is the prediction loss and $\eta$ is the learning rate. $W_Q$ and $W_V$ are updated in the same way. The optimizer also updates embeddings and the rest of the Transformer.

The important distinction is:

```text
W_K and W_V are learned rules stored in the model.
K = XW_K and V = XW_V are results for the current input.
```

### The four calculation steps

For every query position, attention performs:

```text
1. Match:      scores  = Q @ Kᵀ
2. Stabilize:  scores  = scores / √d_head
3. Normalize:  weights = softmax(scores + causal_mask)
4. Retrieve:   output  = weights @ V
```

The complete operation is:

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_h}}+M\right)V$$

$M$ is the causal mask. Future-position scores become $-\infty$, which softmax maps to zero.

### Why causal attention can only read the past

During training, all tokens are processed in parallel. Without a mask, `The` could read `cat` and `sat`, even though those future tokens are supposed to be predicted. That would leak the answers. For three tokens, the allowed connections are:

```text
                         keys being read
                      The    cat    sat
query from The        yes     no     no
query from cat        yes    yes     no
query from sat        yes    yes    yes
```

The mask before softmax is therefore:

```text
[[ 0,  -∞,  -∞],
 [ 0,   0,  -∞],
 [ 0,   0,   0]]
```

After softmax, every masked location has probability zero. Each row still sums to one, so the current token receives a weighted mixture of only itself and its available history.

In [ ]:
def causal_attention(q, k, v):
    d_head = q.size(-1)

    # Each query is compared with every key.
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)

    # True above the diagonal means 'this key is in the future'.
    length = q.size(-2)
    future_mask = torch.triu(
        torch.ones(length, length, dtype=torch.bool), diagonal=1
    )
    scores = scores.masked_fill(future_mask, float('-inf'))

    # Convert scores to probabilities, then retrieve a weighted value.
    weights = F.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights

q = torch.eye(3).unsqueeze(0)
k = q.clone()
v = torch.tensor([[[10., 0.], [0., 20.], [30., 30.]]])

output, weights = causal_attention(q, k, v)
print('Attention weights:\n', weights[0].round(decimals=3))
print('Output:\n', output[0].round(decimals=3))
print('All future weights are zero:', bool(torch.all(weights[0].triu(1) == 0)))

print('\nWhy divide by sqrt(d_head)?')
for d_head in (8, 64, 512):
    random_q = torch.randn(20_000, d_head)
    random_k = torch.randn(20_000, d_head)
    raw = (random_q * random_k).sum(dim=-1)
    scaled = raw / math.sqrt(d_head)
    print(f'Dh={d_head:3d} | raw std={raw.std():6.2f} | scaled std={scaled.std():.2f}')

### Try it yourself

Change `q`, `k`, or `v` below. Predict which output positions will change before running the cell.

In [ ]:
q = torch.eye(3).unsqueeze(0)
k = q.clone()
v = torch.tensor([[[10., 0.], [0., 20.], [30., 30.]]])
output, weights = causal_attention(q, k, v)
print('Weights:\n', weights[0].round(decimals=3))
print('Output:\n', output[0].round(decimals=3))

## 3. Multi-Head Attention: multiple representation subspaces

The key tensor transformation is `[B,T,D] → [B,H,T,Dh]`, where `D = H × Dh`.

In [ ]:
d_model = 8
num_heads = 2
d_head = d_model // num_heads
batch_size, seq_len = 1, 4

x = torch.randn(batch_size, seq_len, d_model)
qkv_projection = nn.Linear(d_model, 3 * d_model, bias=False)
q, k, v = qkv_projection(x).chunk(3, dim=-1)

def split_heads(tensor):
    # [B,T,D] -> [B,T,H,Dh] -> [B,H,T,Dh]
    return tensor.view(batch_size, seq_len, num_heads, d_head).transpose(1, 2)

q, k, v = map(split_heads, (q, k, v))
head_output, head_weights = causal_attention(q, k, v)

# Join the heads: [B,H,T,Dh] -> [B,T,H,Dh] -> [B,T,D]
joined = head_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

print('Input:             ', tuple(x.shape), '[B,T,D]')
print('Q after splitting: ', tuple(q.shape), '[B,H,T,Dh]')
print('Attention weights: ', tuple(head_weights.shape), '[B,H,T,T]')
print('Joined output:     ', tuple(joined.shape), '[B,T,D]')

## 4. RMSNorm: control activation scale

### Why does a Transformer need normalization?

Every Transformer block repeatedly transforms and adds vectors:

```python
x = x + attention(x)
x = x + ffn(x)
```

After many layers, these repeated matrix multiplications, nonlinear functions, and residual additions can make activation magnitudes drift. They may grow rapidly:

```text
layer 1: [ 0.8, -0.4,  1.2,  0.3]
layer 2: [ 2.1, -1.3,  3.7,  0.9]
layer 3: [ 8.4, -5.7, 12.1,  4.3]
```

or shrink toward zero. Uncontrolled scales can create extreme attention logits, saturated nonlinearities, exploding or vanishing gradients, and a model whose useful learning-rate range is very narrow.

RMSNorm gives attention and the FFN inputs with a predictable scale. It does not add contextual information or decide which token to attend to; it provides a numerically stable working range for the components that do.

### What is the root mean square?

For a token vector $x=[x_1,\ldots,x_D]$:

$$\operatorname{RMS}(x)=\sqrt{\frac{1}{D}\sum_{i=1}^{D}x_i^2+\epsilon}$$

For $x=[1,2,3,4]$:

```text
squares: [1, 4, 9, 16]
mean:    (1 + 4 + 9 + 16) / 4 = 7.5
RMS:     √7.5 ≈ 2.739
```

RMSNorm divides the vector by this value:

$$\hat{x}=\frac{x}{\operatorname{RMS}(x)}$$

The normalized vector has an RMS close to 1. Therefore, vectors with the same direction but very different magnitudes become nearly identical:

```text
[  1,   2,   3,   4] ── RMSNorm ──→ [0.365, 0.730, 1.095, 1.461]
[100, 200, 300, 400] ── RMSNorm ──→ [0.365, 0.730, 1.095, 1.461]
```

This removes an uncontrolled overall magnitude while largely preserving the vector's direction.

### Why is there a learnable weight?

Forcing every feature to remain at exactly the same scale would be unnecessarily restrictive. RMSNorm therefore multiplies the normalized vector by a learned vector $g$:

$$y_i=g_i\frac{x_i}{\operatorname{RMS}(x)}$$

The scale starts as `g = [1, 1, ..., 1]`, but training may learn that some feature dimensions should be amplified and others reduced. RMSNorm therefore combines a stable overall scale with learnable per-feature flexibility.

The small $\epsilon$ prevents division by zero when an input vector is zero or extremely small.

### Which dimensions are normalized?

For an input of shape `[B, T, D]`, RMSNorm operates independently on the final `D` features of every token:

```text
x:   [B, T, D]
            └── compute one RMS across these D features
rms: [B, T, 1]
y:   [B, T, D]
```

Tokens and batch items are not averaged together.

### RMSNorm in a modern Transformer

Modern decoder-only models commonly use **Pre-Norm**:

```python
x = x + attention(rms_norm_1(x))
x = x + ffn(rms_norm_2(x))
```

```text
      ┌──────────── direct residual path ────────────┐
      │                                               ↓
x ────┴──→ RMSNorm → Attention or FFN ─────────────→ (+) → new x
```

Attention and the FFN receive scale-controlled inputs, while the residual stream retains a direct identity path. This direct path also helps gradients propagate through deep networks.

Unlike LayerNorm, RMSNorm does not subtract the mean. LayerNorm controls both center and scale; RMSNorm controls scale only. This simpler operation has proved sufficient in many modern language models.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * x / rms

norm = RMSNorm(dim=4)
x = torch.tensor([[1., 2., 3., 4.], [100., 200., 300., 400.]])
y = norm(x)

print('RMS before:', x.pow(2).mean(-1).sqrt())
print('RMS after: ', y.pow(2).mean(-1).sqrt())
print('Normalized vectors are equal:', torch.allclose(y[0], y[1]))

## 5. RoPE: rotate Q/K to represent relative position

### Why attention needs positional information

Attention mainly compares token content. Without positional information, the same tokens in `dog bites man` and `man bites dog` are difficult to distinguish even though their order changes the meaning.

A causal mask only says which positions are visible—it prevents a token from reading the future. It does not clearly encode that one token is one step away while another is five steps away. RoPE supplies this finer relative-position information.

```text
causal mask → may I read this position?
RoPE        → where is it relative to me?
```

### The rotation intuition

RoPE treats each pair of vector dimensions as a point in a 2D plane and rotates it by an angle determined by token position:

```text
same content vector [1, 0]

position 0 → rotate  0° → [1.000, 0.000]
position 1 → rotate 30° → [0.866, 0.500]
position 2 → rotate 60° → [0.500, 0.866]
```

Rotation changes direction but preserves length. The content scale remains stable, while the direction now carries position. Different dimension pairs rotate at different frequencies, allowing the model to represent both short- and long-range position patterns.

### Why relative position appears in Q·K

A query at position $m$ is rotated by $R_m$, and a key at position $n$ is rotated by $R_n$. Their attention score contains:

$$(R_mq)^T(R_nk)=q^TR_{n-m}k$$

The result depends on $n-m$, the relative distance between the two tokens. RoPE therefore injects relative position directly into the content-matching score.

### Why rotate Q and K, but usually not V?

Q and K decide **which token to read**, so rotating them lets position affect the attention weights. V carries **the information being read** and usually does not need rotation.

```text
X ── W_Q → Q → RoPE ─┐
                      ├→ QKᵀ → attention weights
X ── W_K → K → RoPE ─┘

X ── W_V → V ─────────→ weighted information
```

In short, Q/K/V learn what content should match, the causal mask blocks future answers, and RoPE tells the matching process where tokens are relative to one another.

In [ ]:
def apply_rope(x):
    # x has shape [..., sequence_length, dimension].
    length, dim = x.size(-2), x.size(-1)
    assert dim % 2 == 0

    positions = torch.arange(length, dtype=x.dtype)
    inv_freq = 1.0 / (10_000 ** (torch.arange(0, dim, 2) / dim))
    angles = positions[:, None] * inv_freq[None, :]
    cos, sin = angles.cos(), angles.sin()

    even = x[..., 0::2]
    odd = x[..., 1::2]
    rotated_pairs = torch.stack(
        (even * cos - odd * sin, even * sin + odd * cos), dim=-1
    )
    return rotated_pairs.flatten(-2)

x = torch.tensor([[1., 0., 1., 0.]]).repeat(4, 1)
rotated = apply_rope(x)
print('Rotated vectors:\n', rotated.round(decimals=3))
print('Norms before:', x.norm(dim=-1))
print('Norms after: ', rotated.norm(dim=-1))

## 6. SwiGLU: content branch × gate branch

`SiLU(xW_gate) * (xW_up)` lets the model create candidate features and decide how much of each one to pass through.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        gate = F.silu(self.gate(x))
        content = self.up(x)
        return self.down(gate * content)

layer = SwiGLU(d_model=4, d_ff=8)
x = torch.randn(2, 3, 4)
y = layer(x)
print('Input shape: ', tuple(x.shape))
print('Output shape:', tuple(y.shape))

## 7. Assemble a Decoder-Only Transformer

Each block uses Pre-Norm:

```python
x = x + attention(norm(x))
x = x + ffn(norm(x))
```

The direct residual addition provides an unobstructed path for forward information and backward gradients.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        batch, length, d_model = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)

        def split(tensor):
            return tensor.view(
                batch, length, self.num_heads, self.d_head
            ).transpose(1, 2)

        q, k, v = map(split, (q, k, v))
        q, k = apply_rope(q), apply_rope(k)
        attended, _ = causal_attention(q, k, v)
        joined = attended.transpose(1, 2).contiguous().view(batch, length, d_model)
        return self.out(joined)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        # Pre-Norm: normalization stays off the direct residual path.
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x


class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_layers=2, num_heads=4, d_ff=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight  # Weight tying

    def forward(self, token_ids, targets=None):
        x = self.embedding(token_ids)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss


model = TinyLM(vocab_size=20)
tokens = torch.randint(0, 20, (2, 8))
targets = torch.randint(0, 20, (2, 8))
logits, loss = model(tokens, targets)
loss.backward()

print('Token shape: ', tuple(tokens.shape), '[B,T]')
print('Logit shape: ', tuple(logits.shape), '[B,T,V]')
print('Loss:        ', round(loss.item(), 4))
print('Parameter count:', sum(p.numel() for p in model.parameters()))

## 8. Training Diagnostic: overfit a tiny corpus

Overfitting a repeated tiny corpus is an implementation test, not a way to build a useful model. Expect a substantial loss decrease, finite gradient norms, and generated text resembling the corpus.

In [ ]:
text = 'to be or not to be, that is the question. ' * 12
chars = sorted(set(text))
to_id = {char: index for index, char in enumerate(chars)}
to_char = {index: char for char, index in to_id.items()}
data = torch.tensor([to_id[char] for char in text], dtype=torch.long)

seq_len = 16
batch_size = 16
steps = 200
model = TinyLM(
    vocab_size=len(chars), d_model=48, num_layers=2, num_heads=4, d_ff=128
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)

model.train()
for step in range(steps):
    starts = torch.randint(0, len(data) - seq_len - 1, (batch_size,))

    # The target at each position is the next token after the input.
    x = torch.stack([data[s : s + seq_len] for s in starts])
    y = torch.stack([data[s + 1 : s + seq_len + 1] for s in starts])

    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step == 0 or (step + 1) % 50 == 0:
        print(
            f'step {step + 1:3d} | loss {loss.item():.4f} '
            f'| grad_norm {float(grad_norm):.3f}'
        )

# Greedy generation: repeatedly choose the highest-scoring next token.
model.eval()
generated = torch.tensor([[to_id[c] for c in 'to ']], dtype=torch.long)
with torch.no_grad():
    for _ in range(40):
        logits, _ = model(generated)
        next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
        generated = torch.cat((generated, next_id), dim=1)

result = ''.join(to_char[index] for index in generated[0].tolist())
print('Generated text:', result)

## Next Experiments

1. Remove the causal mask and inspect future-position weights.
2. Remove `sqrt(Dh)` and compare the scale of attention logits.
3. Change two heads to four and record the shape changes.
4. Remove RMSNorm temporarily and compare loss and gradient norms.
5. Replace SwiGLU with `Linear → GELU → Linear`.
6. Stop shifting the targets and explain why the model learns to copy.